# [SCENE] ROTBOTS -- Video Plan

---

Configure your video before building it. This plan controls everything downstream.

> **[?]** You're about to design a content machine. What decisions are you making vs. the machine?

---

In [ ]:
# SETUP
import json, re
from pathlib import Path
from IPython.display import display, Markdown
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
BASE_DIR.mkdir(parents=True, exist_ok=True)
print('[OK] Setup')

---
## Video Plan

Set the total length, content mix, and features for your video.


In [ ]:
# =============================================
#   YOUR VIDEO PLAN -- edit everything below
# =============================================

TOPIC = 'The history and ethics of AI-generated art'

SOURCES = [
    'https://en.wikipedia.org/wiki/Artificial_intelligence_art',
    # --- Add more sources below (uncomment or add your own) ---
    # 'https://example.com/any-article-or-blog-post',
    # 'https://arxiv.org/pdf/2302.06576',              # PDF link
    # 'https://www.nytimes.com/2023/ai-art-debate',     # news article
    # 'https://medium.com/@someone/my-essay-123',       # blog post
    # 'Raw text works too: just paste your text as a string.',
]

# ---- Video length & scene settings ----
TOTAL_VIDEO_LENGTH = 60    # total seconds (including credits)
SECONDS_PER_SCENE  = 5    # seconds per scene/clip

# ---- Content mix (must add up to <= 1.0, rest = AI-generated) ----
ARCHIVE_RATIO = 0.0       # 0.0-1.0 -- archive.org footage
UPLOAD_RATIO  = 0.0       # 0.0-1.0 -- your own uploaded footage

# ---- Features (True/False) ----
ENABLE_CREDITS   = True
ENABLE_SUBTITLES = False
ENABLE_MUSIC     = False
ENABLE_EFFECTS   = True

SESSION_NAME = ''  # leave empty to auto-generate from topic


In [ ]:
# CALCULATE PLAN
AI_RATIO = max(0, 1.0 - ARCHIVE_RATIO - UPLOAD_RATIO)
if AI_RATIO + ARCHIVE_RATIO + UPLOAD_RATIO > 1.0:
    print('[!!] Ratios exceed 1.0!'); AI_RATIO=1.0; ARCHIVE_RATIO=0; UPLOAD_RATIO=0
CREDITS_LENGTH = 8 if ENABLE_CREDITS else 0
CONTENT_LENGTH = TOTAL_VIDEO_LENGTH - CREDITS_LENGTH
NARRATION_LENGTH = CONTENT_LENGTH
NUM_TOTAL_SCENES = max(2, int(CONTENT_LENGTH / SECONDS_PER_SCENE))
WORDS_PER_SCENE = SECONDS_PER_SCENE * 2.5

# Scene counts: 0-ratio = 0 scenes, AI gets remainder
NUM_ARCHIVE_SCENES = int(NUM_TOTAL_SCENES * ARCHIVE_RATIO) if ARCHIVE_RATIO > 0 else 0
NUM_UPLOAD_SCENES = int(NUM_TOTAL_SCENES * UPLOAD_RATIO) if UPLOAD_RATIO > 0 else 0
NUM_AI_SCENES = max(1, NUM_TOTAL_SCENES - NUM_ARCHIVE_SCENES - NUM_UPLOAD_SCENES)
while NUM_AI_SCENES + NUM_ARCHIVE_SCENES + NUM_UPLOAD_SCENES > NUM_TOTAL_SCENES:
    if NUM_ARCHIVE_SCENES > 0: NUM_ARCHIVE_SCENES -= 1
    elif NUM_UPLOAD_SCENES > 0: NUM_UPLOAD_SCENES -= 1
    else: break

NUM_CHAPTERS = max(1, min(3, NUM_TOTAL_SCENES // 3))
SECTIONS_PER_CHAPTER = max(1, NUM_TOTAL_SCENES // NUM_CHAPTERS)

# Interleave scene order (proportional-need algorithm)
scene_types = []
ai_p = 0; ar_p = 0; up_p = 0
for i in range(NUM_TOTAL_SCENES):
    remaining = max(1, NUM_TOTAL_SCENES - i)
    ai_need = (NUM_AI_SCENES - ai_p) / remaining
    ar_need = (NUM_ARCHIVE_SCENES - ar_p) / remaining
    up_need = (NUM_UPLOAD_SCENES - up_p) / remaining
    if ar_need >= ai_need and ar_need >= up_need and ar_p < NUM_ARCHIVE_SCENES:
        scene_types.append('archive'); ar_p += 1
    elif up_need >= ai_need and up_p < NUM_UPLOAD_SCENES:
        scene_types.append('upload'); up_p += 1
    else:
        scene_types.append('ai_generated'); ai_p += 1

if not SESSION_NAME:
    SESSION_NAME = '-'.join(re.sub(r'[^a-zA-Z0-9 ]', '', TOPIC.lower()).split()[:4])
SESSION_DIR = BASE_DIR / SESSION_NAME
for d in ['', 'videos', 'audio', 'archive_clips', 'uploads']:
    (SESSION_DIR / d).mkdir(parents=True, exist_ok=True)

plan = dict(topic=TOPIC, sources=SOURCES, session_name=SESSION_NAME,
    total_video_length=TOTAL_VIDEO_LENGTH, seconds_per_scene=SECONDS_PER_SCENE,
    content_length=CONTENT_LENGTH, credits_length=CREDITS_LENGTH,
    narration_length=NARRATION_LENGTH, ai_ratio=AI_RATIO,
    archive_ratio=ARCHIVE_RATIO, upload_ratio=UPLOAD_RATIO,
    num_total_scenes=NUM_TOTAL_SCENES, num_ai_scenes=NUM_AI_SCENES,
    num_archive_scenes=NUM_ARCHIVE_SCENES, num_upload_scenes=NUM_UPLOAD_SCENES,
    words_per_scene=WORDS_PER_SCENE, scene_types=scene_types,
    num_chapters=NUM_CHAPTERS, sections_per_chapter=SECTIONS_PER_CHAPTER,
    enable_credits=ENABLE_CREDITS, enable_subtitles=ENABLE_SUBTITLES,
    enable_music=ENABLE_MUSIC, enable_effects=ENABLE_EFFECTS)
with open(SESSION_DIR / 'video_plan.json', 'w') as f: json.dump(plan, f, indent=2)

print(f'Session: {SESSION_NAME}')
print(f'Plan: {TOTAL_VIDEO_LENGTH}s = {CONTENT_LENGTH}s content + {CREDITS_LENGTH}s credits')
print(f'   AI: {NUM_AI_SCENES} scenes')
if NUM_ARCHIVE_SCENES > 0: print(f'   Archive: {NUM_ARCHIVE_SCENES} scenes')
if NUM_UPLOAD_SCENES > 0: print(f'   Upload: {NUM_UPLOAD_SCENES} scenes')
print(f'   Narration: {NARRATION_LENGTH:.0f}s ({int(NARRATION_LENGTH*2.5)} words)')
print(f'\nScene order: {scene_types}')
print(f'\n[OK] Plan saved')

---
*ROTBOTS Workshop -- LI-MA 2026*